# Read the Bronze data and Create a DataFrame

In [1]:
customers_raw = spark.read.parquet("abfss://ecommerce_project@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/customers.parquet")
orders_raw = spark.read.parquet("abfss://ecommerce_project@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/orders.parquet")
payments_raw = spark.read.parquet("abfss://ecommerce_project@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/payments.parquet")
supportticket_raw = spark.read.parquet("abfss://ecommerce_project@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/support_tickets.parquet")
webactivities_raw = spark.read.parquet("abfss://ecommerce_project@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/web_activities.parquet")

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 3, Finished, Available, Finished)

# Reading Bronze Data

In [2]:
display(customers_raw.limit(5))

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, a503c2d8-9982-4e64-8e4f-c9cf0d634abb)

In [3]:
display(orders_raw.limit(5))

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 922b41d1-bed0-4497-96a9-8667d374734b)

In [4]:
display(payments_raw.limit(5))

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 6, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, cc496547-3574-49e0-a221-99f2946e0208)

In [5]:
display(supportticket_raw.limit(5))

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 7, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 1982a2d5-9829-44d3-a95e-ab12b329a416)

In [6]:
display(webactivities_raw.limit(5))

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 8, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, fe70fcca-03f6-44ef-9b37-349f55afa5ba)

# Create Delta bronze Table

In [7]:
# Save as Bronze Delta Table
customers_raw.write.format("delta").mode("overwrite").saveAsTable("customers")
orders_raw.write.format("delta").mode("overwrite").saveAsTable("orders")
payments_raw.write.format("delta").mode("overwrite").saveAsTable("payments")
supportticket_raw.write.format("delta").mode("overwrite").saveAsTable("supportticket")
webactivities_raw.write.format("delta").mode("overwrite").saveAsTable("webactivities")

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 9, Finished, Available, Finished)

# Cleaned data - Silver

In [8]:
display(customers_raw.limit(4))

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 10, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 733759dc-db50-493e-aa86-f351784ec74e)

### Customers data cleaned

In [9]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

customers_clean = (
    customers_raw
    .withColumn("email", lower(trim(col("EMAIL"))))
    .withColumn("name", initcap(trim(col("name"))))
    .withColumn("gender", when(lower(col("gender")).isin("f", "female"), "Female")
                            .when(lower(col("gender")).isin("m", "male"), "Male")
                            .otherwise("Other"))
    .withColumn("dob", to_date(regexp_replace(col("dob"), "/", "-")))
    .withColumn("location", initcap(col("location")))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id", "email"])
)
customers_clean.write.format("delta").mode("overwrite").saveAsTable("silver_Customers")

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 11, Finished, Available, Finished)

In [10]:
display(customers_clean.limit(6))

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 12, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 1948c1fb-5a9c-46d9-9b2d-f0c0f541e413)

### Clean orders table

In [11]:
%%sql

select * from orders

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 13, Finished, Available, Finished)

<Spark SQL result set with 15 rows and 5 fields>

In [12]:
orders = spark.table("orders")
orders_clean = (
    orders
    .withColumn("order_date",
                when(col("order_date").rlike("^\d{4}/\d{2}/\d{2}$"), to_date(col("order_date"), "yyyy/MM/dd"))
                .when(col("order_date").rlike("^\d{2}-\d{2}-\d{4}$"), to_date(col("order_date"), "dd-MM-yyyy"))
                .when(col("order_date").rlike("^\d{2}/\d{2}/\d{4}$"), to_date(col("order_date"), "dd/MM/yyyy"))
                .when(col("order_date").rlike("^\d{8}$"), to_date(col("order_date"), "yyyyMMdd"))
                .otherwise(to_date(col("order_date"), "yyyy-MM-dd")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .withColumn("status", initcap(col("status")))
    .dropna(subset=["customer_id", "order_date"])
    .dropDuplicates(["order_id"])
)
orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 14, Finished, Available, Finished)

In [13]:
display(orders_clean)

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 15, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, ea61474a-cf84-4a61-a15d-35b28999dc6a)

### Payments table - cleaned

In [14]:
%%sql
SELECT * from payments limit 5

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 16, Finished, Available, Finished)

<Spark SQL result set with 5 rows and 6 fields>

In [15]:
payments = spark.table("payments")
payments_clean = (
    payments
    .withColumn("payment_date", to_date(regexp_replace(col("payment_date"), "/", "-")))
    .withColumn("payment_method", initcap(col("payment_method")))
    .replace({"creditcard" : "Credit Card"}, subset=["payment_method"])
    .withColumn("payment_status", initcap(col("payment_status")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .dropna(subset=["customer_id", "payment_date", "amount"])
)
payments_clean.write.format("delta").mode("overwrite").saveAsTable("silver_payments")

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 17, Finished, Available, Finished)

In [16]:
display(payments_clean.limit(5))

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 18, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 3bb9fff9-38f0-4e0f-a7de-e909bb1aafe4)

### Clean Support table

In [17]:
%%sql
SELECT * FROM supportticket LIMIT 5

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 19, Finished, Available, Finished)

<Spark SQL result set with 5 rows and 5 fields>

In [18]:
support = spark.table("supportticket")
support_clean = (
    support
    .withColumn("ticket_date", to_date(regexp_replace(col("ticket_date"), "/", "-")))
    .withColumn("issue_type", initcap(trim(col("issue_type"))))
    .withColumn("resolution_status", initcap(trim(col("resolution_status"))))
    .replace({"NA": None, "": None},subset=["issue_type", "resolution_status"])
    .dropDuplicates(["ticket_id"])
    .dropna(subset=["customer_id", "ticket_date"])
)

support_clean.write.format("delta").mode("overwrite").saveAsTable("silver_support")

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 20, Finished, Available, Finished)

In [19]:
display(support_clean.limit(4))

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 21, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, a932e837-f86f-444b-96a8-29923ca0b2d7)

### Web Data - Clean

In [20]:
%%sql
SELECT * FROM webactivities limit 5

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 22, Finished, Available, Finished)

<Spark SQL result set with 5 rows and 5 fields>

In [21]:
web = spark.table("webactivities")
web_clean = (
    web
    .withColumn("session_time", to_date(regexp_replace(col("session_time"), "/", "-")))
    .withColumn("page_viewed", lower(col("page_viewed")))
    .withColumn("device_type", initcap(col("device_type")))
    .dropDuplicates(["session_id"])
    .dropna(subset=["customer_id", "session_time", "page_viewed"])
)
web_clean.write.format("delta").mode("overwrite").saveAsTable("silver_web")

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 23, Finished, Available, Finished)

In [22]:
display(web_clean.limit(4))

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 24, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 23573aa0-eb52-47f5-84e7-6880040ebe82)

# Gold Tables - Aggregate Tables

In [23]:
cust = spark.table("silver_customers").alias("c")
orders = spark.table("silver_orders").alias("o")
payments = spark.table("silver_payments").alias("p")
support = spark.table("silver_support").alias("s")
web = spark.table("silver_web").alias("w")

customer360 = (
    cust
    .join(orders, "customer_id", "left")
    .join(payments, "customer_id", "left")
    .join(support, "customer_id", "left")
    .join(web, "customer_id", "left")
    .select(
        col("c.customer_id"),
        col("c.name"), 
        col("c.email"), 
        col("c.gender"), 
        col("c.dob"), 
        col("c.location"),

        col("o.order_id"), 
        col("o.order_date"), 
        col("o.amount").alias("order_amount"), 
        col("o.status").alias("order_status"),

        col("p.payment_method"), 
        col("p.payment_status"), 
        col("p.amount"),

        col("s.ticket_id"), 
        col("s.issue_type"), 
        col("s.ticket_date"), 
        col("s.resolution_status"),

        col("w.page_viewed"), 
        col("w.device_type"), 
        col("w.session_time")
    )
)
customer360.write.format("delta").mode("overwrite").saveAsTable("gold_customer360")

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 25, Finished, Available, Finished)

In [24]:
display(customer360)

StatementMeta(, e9834dbf-c553-4d7a-835e-7fe65a607923, 26, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 3ad0429a-eba6-4c8c-ad51-70d9785d8bed)